# 🧩 Mini-Lab: Extract Your First Pure Function

**Module: The Python That Runs Your Pipeline** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Explain** what makes a function "pure" — all inputs as parameters, no side effects, deterministic output
2. **Extract** inline notebook logic into a pure function that takes parameters and returns a result
3. **Recognize** the difference between a function that secretly depends on globals and one that is fully self-contained

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Pure Function | A pure function takes inputs only through its parameters and produces results only via its return value, printing and side effects are the caller's job. |
| Notebook vs Reusable Code | Notebook smells happen when hidden cell-order state, globals, and magic constants make code work only in one run, not when reused in a pipeline or module. |


## What Is a Pure Function?

A **pure function** follows two rules:

1. **Same inputs → same output.** It reads nothing except its parameters.
2. **No side effects.** It doesn't modify globals, write files, or change things outside itself.

Why does this matter for AI engineering?

- You can **import** a pure function into a script, a FastAPI endpoint, or a test — it just works.
- You can **test** it by calling it with known inputs and checking the output.
- You can **reuse** it across notebooks without worrying about hidden state.

## The Problem: Notebook-Style Code

Recall from the previous mini-lab that notebook code often relies on globals. Here's a typical example — inline logic that scores documents by keyword overlap:

In [1]:
# --- Notebook-style: globals everywhere ---
query = "How do transformers handle long sequences?"
documents = [
    "Transformers use attention to process sequences in parallel.",
    "RNNs handle sequences one token at a time.",
    "Positional encoding tells transformers about word order.",
]
threshold = 0.3

# Inline scoring logic
query_words = set(query.lower().split())
scored = []
for doc in documents:
    doc_words = set(doc.lower().split())
    overlap = len(query_words & doc_words) / len(query_words)
    if overlap >= threshold:
        scored.append({"text": doc, "score": round(overlap, 2)})

scored.sort(key=lambda x: x["score"], reverse=True)
print("Matching documents:")
for item in scored:
    print(f"  [{item['score']}] {item['text']}")

Matching documents:


This works — but it has the smells we identified last time:

- `query`, `documents`, and `threshold` are **globals** — the logic can't run without them.
- The scoring logic is **inline** — you can't call it from another file or test it in isolation.
- If you change `threshold` three notebooks later, this cell silently changes behavior.

Let's fix this by extracting a pure function.

## Step 1: Identify Inputs and Output

Before writing the function, ask:

- **What does this logic need?** → a query, a list of documents, and a threshold
- **What does it produce?** → a sorted list of matching documents with scores

Every piece of data the logic touches becomes a **parameter**. The result becomes the **return value**.

## Step 2: Extract the Pure Function

We move the logic into a function where *everything* it needs comes through parameters:

In [2]:
def score_documents(query, documents, threshold=0.3):
    """Score documents by keyword overlap with the query.

    Returns a sorted list of dicts with 'text' and 'score' keys,
    including only documents at or above the threshold.
    """
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append({"text": doc, "score": round(overlap, 2)})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

Notice what changed:

| Before (notebook-style) | After (pure function) |
|------------------------|----------------------|
| `query` is a global | `query` is a parameter |
| `documents` is a global | `documents` is a parameter |
| `threshold` is a global | `threshold` is a parameter with a default |
| Results printed inline | Results **returned** |
| Can't reuse or test | Can import, call, and test anywhere |

## Step 3: Call It Like Any Other Function

Now the caller decides what data to pass — no globals needed:

In [3]:
documents = [
    "Transformers use attention to process sequences in parallel.",
    "RNNs handle sequences one token at a time.",
    "Positional encoding tells transformers about word order.",
]

# Call with one query
results = score_documents("How do transformers handle long sequences?", documents)
print("Matches:")
for r in results:
    print(f"  [{r['score']}] {r['text']}")

# Call again with a stricter threshold — same function, different config
strict = score_documents("How do transformers handle long sequences?", documents, threshold=0.4)
print(f"\nWith threshold=0.4: {len(strict)} match(es)")

Matches:

With threshold=0.4: 0 match(es)


The function is **pure**: same inputs always produce the same output, and it touches nothing outside itself.

## The Purity Test

A quick way to check if your function is pure — it should pass all three:

| Test | Question | `score_documents` |
|------|----------|-------------------|
| **No globals** | Does it read any variable not in its parameter list? | ✅ No |
| **No side effects** | Does it modify anything outside itself (globals, files, prints)? | ✅ No |
| **Deterministic** | Same arguments → same return value, every time? | ✅ Yes |

## Impure vs. Pure — Side by Side

Let's see one more quick comparison to make the pattern stick. Here's a common notebook pattern that *looks* like a function but is actually impure:

In [4]:
# --- IMPURE: secretly reads a global ---
MODEL = "gpt-4o-mini"

def build_request_impure(prompt):
    return {"model": MODEL, "messages": [{"role": "user", "content": prompt}]}


# --- PURE: everything comes through parameters ---
def build_request_pure(prompt, model):
    return {"model": model, "messages": [{"role": "user", "content": prompt}]}


# The pure version works without any globals
print("Impure:", build_request_impure("Hello"))
print("Pure:  ", build_request_pure("Hello", "gpt-4o-mini"))

Impure: {'model': 'gpt-4o-mini', 'messages': [{'role': 'user', 'content': 'Hello'}]}
Pure:   {'model': 'gpt-4o-mini', 'messages': [{'role': 'user', 'content': 'Hello'}]}


Both produce the same dict *right now* — but only the pure version survives being imported into a different file where `MODEL` doesn't exist.

## 🎯 Summary

### Key Takeaways

1. **Pure function** — a function whose output depends only on its arguments and that has no side effects; it's the smallest unit of reusable code
2. **Extract by asking two questions** — "What does this logic need?" (→ parameters) and "What does it produce?" (→ return value)
3. **Globals become parameters** — every external variable the logic touches should be passed in explicitly, not read from the notebook's global scope
4. **Return, don't print** — a pure function returns its result so the caller decides what to do with it (print, store, test, pipe to an API)